In [0]:
resource_path = '/Volumes/workspace/pj_sales/sales_resources/origin/'
processed_path = '/Volumes/workspace/pj_sales/sales_resources/processed/'

In [0]:
%sql
describe workspace.pj_sales.tb_sales_bronze

In [0]:
%sql
create or replace temp view vw_sales_silver_clean as (
  with cleaned as (
    select
       id_nf                                  as invoice_id
      ,data_venda                             as sale_date
      ,cliente_id                             as client_id
      ,cliente_nome                           as client_name
      ,marca_carro                            as car_brand
      ,modelo_carro                           as car_model
      ,produto_peca                           as car_part
      ,valor_unitario                         as part_unit_price
      ,zeroifnull(quantidade)                 as part_quantity
      ,custo_unitario                         as part_unit_cost
      ,loja_id                                as store_id
      ,loja_nome                              as store_name
      ,case
        when grupo_loja = 'Grupo Atlântic'    then 'Grupo Atlantico'
        when grupo_loja = 'Grupo Atlantico'   then 'Grupo Atlantico'
        when grupo_loja = 'Grupo Pampas'      then 'Grupo Pampa'
        when grupo_loja = 'Grupo Pampa'       then 'Grupo Pampa'
        when grupo_loja = 'Grupo Serras'      then 'Grupo Serra'
        when grupo_loja = 'Grupo Serra'       then 'Grupo Serra'
        when grupo_loja = 'Grupo Tytan'       then 'Grupo Titan'
        when grupo_loja = 'Grupo Titan'       then 'Grupo Titan'
       end                                    as store_group
      ,vendedor_id                            as seller_id
      ,vendedor_nome                          as seller_name
      ,ingestion_timestamp
      ,source_file
    from workspace.pj_sales.tb_sales_bronze
    where id_nf                               is not null
      and zeroifnull(quantidade)              >= 0 -- Os valores negativos ou iguais a 0 é necessário serem tirados?
      and custo_unitario                      > 0 -- Custos e valores menores do que 0 são alguma regra de negócio?
      and valor_unitario                      > 0 -- *
  )
  select
       trim(upper(invoice_id))                as invoice_id -- Há diferença de id em tamanho da string?
      ,cast(sale_date as date)                as sale_date
      ,client_id
      ,trim(lower(client_name))               as client_name
      ,trim(lower(car_brand))                 as car_brand
      ,trim(lower(car_model))                 as car_model
      ,trim(lower(car_part))                  as car_part
      ,part_unit_price
      ,cast(part_quantity as int)             as part_quantity
      ,part_unit_cost
      ,store_id
      ,trim(lower(store_name))                as store_name
      ,trim(lower(store_group))               as store_group
      ,trim(upper(seller_id))                 as seller_id -- Há diferença de id em tamanho da string?
      ,trim(lower(seller_name))               as seller_name
      ,ingestion_timestamp
      ,source_file
  from cleaned
)


In [0]:
%sql
create or replace temp view vw_sales_silver_dedup as (
  with dedup as (
    select
       invoice_id
      ,sale_date
      ,client_id
      ,client_name
      ,car_brand
      ,car_model
      ,car_part
      ,part_unit_price
      ,part_quantity
      ,part_unit_cost
      ,store_id
      ,store_name
      ,store_group
      ,seller_id
      ,seller_name
      ,ingestion_timestamp
      ,source_file
      ,row_number() over 
        (partition by invoice_id 
          order by 
             sale_date             desc 
            ,ingestion_timestamp   desc
        )                          as order_by_date
    from vw_sales_silver_clean
  )
  select
     invoice_id
    ,sale_date
    ,client_id
    ,client_name
    ,car_brand
    ,car_model
    ,car_part
    ,part_quantity
    ,part_unit_price
    ,part_unit_cost
    ,round(part_unit_price * part_quantity, 2)                      as part_total_price
    ,round(part_unit_cost * part_quantity, 2)                       as part_total_cost
    ,round(part_total_price - part_total_cost, 2)                   as part_profit
    ,store_id
    ,store_name
    ,store_group
    ,seller_id
    ,seller_name
    ,ingestion_timestamp
    ,from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')   as updating_timestamp
    ,source_file
  from dedup
  where order_by_date = 1
)


In [0]:
%sql
create table if not exists workspace.pj_sales.tb_sales_silver
using delta
as
select
  *
from vw_sales_silver_dedup

In [0]:
%sql
select * from workspace.pj_sales.tb_sales_silver

In [0]:
%sql
merge into workspace.pj_sales.tb_sales_silver s
using vw_sales_silver_dedup sd
on s.invoice_id = sd.invoice_id
when matched then
  update set
     s.invoice_id            = sd.invoice_id
    ,s.sale_date             = sd.sale_date
    ,s.client_id             = sd.client_id
    ,s.client_name           = sd.client_name
    ,s.car_brand             = sd.car_brand
    ,s.car_model             = sd.car_model
    ,s.car_part              = sd.car_part
    ,s.part_unit_price       = sd.part_unit_price
    ,s.part_quantity         = sd.part_quantity
    ,s.part_unit_cost        = sd.part_unit_cost
    ,s.part_total_price      = sd.part_total_price
    ,s.part_total_cost       = sd.part_total_cost
    ,s.part_profit           = sd.part_profit
    ,s.store_id              = sd.store_id
    ,s.store_name            = sd.store_name
    ,s.store_group           = sd.store_group
    ,s.seller_id             = sd.seller_id
    ,s.seller_name           = sd.seller_name
    ,s.updating_timestamp    = from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')
    ,s.source_file           = sd.source_file -- Atualização de dados veio de um novo arquivo, arquivo antigo passa a ser desconsiderado?
when not matched then
  insert *